In [0]:
# Databricks notebook source
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import IntegerType, DoubleType

spark = SparkSession.builder.getOrCreate()



In [0]:
bronze = spark.table("workspace.default.capstone_bronze_sales")

df = (bronze
      .withColumn("order_id", F.trim(F.col("order_id").cast("string")))
      .withColumn("order_date", F.to_date(F.col("order_date")))
      .withColumn("customer_id", F.trim(F.col("customer_id").cast("string")))
      .withColumn("customer_name", F.trim(F.col("customer_name")))
      .withColumn("city", F.trim(F.col("city")))
      .withColumn("state", F.trim(F.col("state")))
      .withColumn("product_id", F.trim(F.col("product_id")))
      .withColumn("product_name", F.trim(F.col("product_name")))
      .withColumn("category", F.trim(F.col("category")))
      .withColumn("quantity", F.col("quantity").cast(IntegerType()))
      .withColumn("unit_price", F.col("unit_price").cast(DoubleType()))
      .withColumn("discount_pct", F.col("discount_pct").cast(DoubleType()))
      .withColumn("gross_amount", F.col("gross_amount").cast(DoubleType()))
      .withColumn("discount_amount", F.col("discount_amount").cast(DoubleType()))
      .withColumn("net_amount", F.col("net_amount").cast(DoubleType()))
      .withColumn("payment_method", F.trim(F.col("payment_method")))
      .withColumn("order_status", F.trim(F.col("order_status"))))

In [0]:
df = df.withColumn(
    "quality_flag",
    F.when(F.col("customer_id").isNull(), "INVALID_CUSTOMER")
     .when(F.col("quantity").isNull() | (F.col("quantity") <= 0), "INVALID_QUANTITY")
     .when(F.col("net_amount").isNull() | (F.col("net_amount") < 0), "INVALID_AMOUNT")
     .when(F.col("product_id").isNull() | (F.upper(F.col("product_id")) == "UNKNOWN"), "INVALID_PRODUCT")
     .otherwise("VALID"))

total_count = df.count()

df_valid = (df.filter(F.col("quality_flag") == "VALID")
            .dropDuplicates(["order_id"]))

In [0]:
df_silver = (df_valid
             .withColumn("year", F.year("order_date"))
             .withColumn("month", F.month("order_date"))
             .withColumn("month_name", F.date_format("order_date", "MMMM"))
             .drop("quality_flag"))

df_silver.write.format("delta").mode("overwrite") \
    .saveAsTable("workspace.default.capstone_silver_sales")

print(f"[SILVER] {df_silver.count()} valid records kept out of {total_count} total")